# Assignment 4 - Survival Analysis

This notebook walks through the three required survival analysis methods:
1. Kaplan-Meier Analysis
2. Cox Proportional Hazards Model
3. Random Survival Forest

**You can complete the assignment by:**
- Implementing functions in `students/` modules AND running this notebook, OR
- Writing a custom script that calls your implemented functions

**Both approaches are valid** as long as you generate all required outputs.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# Import student implementations
from students import (
    fit_kaplan_meier,
    compute_logrank_test,
    plot_km_curves,
    fit_cox_model,
    extract_cox_summary,
    test_proportional_hazards,
    fit_random_survival_forest,
    compute_concordance_index,
    get_feature_importance,
    plot_feature_importance,
    set_plot_style,
)

# Set plotting style
set_plot_style()

# Create outputs directory
Path('outputs').mkdir(exist_ok=True)

## 1. Load Data

Load the RADCURE clinical dataset.

**Download instructions:**
- Option 1: From TCIA website (see data/README.md)
- Option 2: From Blackboard assignment folder

The file should be saved as `data/RADCURE-clinical-data.xlsx` or `.csv`

In [ ]:
# Load RADCURE clinical data
# If you have the Excel file:
data = pd.read_excel('../data/RADCURE-clinical-data.xlsx')

# If you converted to CSV:
# data = pd.read_csv('../data/RADCURE-clinical-data.csv')

# Display basic info
print(f"Dataset shape: {data.shape}")
print(f"\nFirst few column names: {list(data.columns[:10])}")
print(f"\nData types:\n{data.dtypes.value_counts()}")

# Show first few rows
data.head()

In [ ]:
# Prepare survival variables
# RADCURE column names may vary - check your file!
time_col = 'Survival_time_in_days'  # Or check actual column name
event_col = 'Death'  # 1 = died, 0 = alive

# Verify columns exist
print("Available columns related to survival:")
print([col for col in data.columns if 'time' in col.lower() or 'death' in col.lower() or 'survival' in col.lower()])

# Check event distribution
print(f"\nEvent distribution ({event_col}):")
print(data[event_col].value_counts())
print(f"\nCensoring rate: {(1 - data[event_col].mean())*100:.1f}%")

# Summary statistics
print(f"\nSurvival time (days):")
print(data[time_col].describe())

## 2. Kaplan-Meier Analysis

Compare survival curves between patient groups.

In [ ]:
# Create grouping variable for KM analysis
# Option 1: Age groups
data['Age_Group'] = pd.cut(data['Age'], bins=[0, 60, 100], labels=['≤60', '>60'])

# Option 2: HPV status (if available)
# group_col = 'HPV_Combined'

# Option 3: Overall stage (Early vs Advanced)
# data['Stage_Group'] = data['Overall_Stage'].map({
#     'I': 'Early', 'II': 'Early',
#     'III': 'Advanced', 'IV': 'Advanced'
# })

# Choose which grouping to use
group_col = 'Age_Group'

print(f"Grouping by: {group_col}")
print(data[group_col].value_counts())

In [ ]:
# Fit KM curves
km_models = fit_kaplan_meier(
    data=data,
    time_col=time_col,
    event_col=event_col,
    group_col=group_col
)

print(f"Fitted KM curves for {len(km_models)} groups: {list(km_models.keys())}")

In [ ]:
# Compute log-rank test
logrank_results = compute_logrank_test(
    data=data,
    time_col=time_col,
    event_col=event_col,
    group_col=group_col
)

print("\nLog-rank test results:")
print(f"  Test statistic: {logrank_results['test_statistic']:.4f}")
print(f"  p-value: {logrank_results['p_value']:.4f}")

if logrank_results['p_value'] < 0.05:
    print("  → Survival curves are significantly different")
else:
    print("  → No significant difference between groups")

In [ ]:
# Save log-rank results
with open('outputs/logrank_results.json', 'w') as f:
    json.dump(logrank_results, f, indent=2)

print("✅ Saved: outputs/logrank_results.json")

In [ ]:
# Generate KM plot
plot_km_curves(
    km_models=km_models,
    filename='outputs/km_survival_plot.png',
    title=f'Kaplan-Meier Survival Curves by {group_col}'
)

print("✅ Saved: outputs/km_survival_plot.png")

## 3. Cox Proportional Hazards Model

Fit regression model with ≥3 covariates.

In [ ]:
# Select covariates (must have at least 3)
# RADCURE example covariates:
covariates = [
    'Age',                    # Continuous
    'Gender',                 # Categorical
    'Overall_Stage',          # Categorical (I, II, III, IV)
    'HPV_Combined',           # Categorical (if available)
    'Chemotherapy',           # Binary (Yes/No)
    'RT_Dose_Gy'              # Continuous
]

# Verify columns exist and handle encoding
print("Checking covariates...")
available_covariates = [col for col in covariates if col in data.columns]
print(f"Available: {available_covariates}")

# For categorical variables, may need to encode
# Lifelines handles this automatically, but check data types
for col in available_covariates:
    print(f"{col}: {data[col].dtype}, unique values: {data[col].nunique()}")

# Use at least 3 available covariates
covariates = available_covariates[:6]  # Use first 6 if available
print(f"\nUsing {len(covariates)} covariates: {covariates}")

In [ ]:
# Fit Cox model
cox = fit_cox_model(
    data=data,
    time_col=time_col,
    event_col=event_col,
    covariates=covariates
)

print("✅ Cox model fitted")

In [ ]:
# Extract summary
cox_summary = extract_cox_summary(cox)

print("\nCox Model Summary:")
print(cox_summary)

In [ ]:
# Save Cox summary
cox_summary.to_csv('outputs/cox_summary.csv', index=False)
print("✅ Saved: outputs/cox_summary.csv")

In [ ]:
# Test proportional hazards assumption
ph_test = test_proportional_hazards(
    cox_model=cox,
    data=data,
    time_col=time_col,
    event_col=event_col
)

print("\nProportional Hazards Test:")
for covar, result in ph_test.items():
    status = "✓" if result['p_value'] > 0.05 else "✗"
    print(f"  {status} {covar}: p = {result['p_value']:.4f}")

In [ ]:
# Save PH test results
with open('outputs/cox_ph_test.json', 'w') as f:
    json.dump(ph_test, f, indent=2)

print("✅ Saved: outputs/cox_ph_test.json")

## 4. Random Survival Forest

Train ensemble model and evaluate performance.

In [ ]:
# Prepare features
from sksurv.util import Surv

# Select features (can be same as Cox covariates or different)
feature_cols = covariates
X = data[feature_cols].copy()

# Prepare survival outcome
y = Surv.from_dataframe('event', 'time', data)

print(f"Feature matrix shape: {X.shape}")
print(f"Survival outcome shape: {y.shape}")

In [ ]:
# Train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

In [ ]:
# Fit Random Survival Forest
rsf = fit_random_survival_forest(
    X_train=X_train,
    y_train=y_train,
    n_estimators=100,
    random_state=42
)

print("✅ Random Survival Forest trained")

In [ ]:
# Compute C-index
c_index = compute_concordance_index(rsf, X_test, y_test)

print(f"\nConcordance Index (C-index): {c_index:.4f}")
print("\nInterpretation:")
if c_index > 0.7:
    print("  → Good predictive performance")
elif c_index > 0.6:
    print("  → Moderate predictive performance")
else:
    print("  → Weak predictive performance")

In [ ]:
# Save metrics
rsf_metrics = {'c_index': float(c_index)}

with open('outputs/rsf_metrics.json', 'w') as f:
    json.dump(rsf_metrics, f, indent=2)

print("✅ Saved: outputs/rsf_metrics.json")

In [ ]:
# Get feature importance
importance = get_feature_importance(rsf, feature_cols)

print("\nFeature Importance:")
print(importance)

In [ ]:
# Plot feature importance
plot_feature_importance(
    importance_df=importance,
    filename='outputs/rsf_importance.png',
    top_n=10
)

print("✅ Saved: outputs/rsf_importance.png")

## 5. Verify All Outputs

Check that all required files were generated.

In [ ]:
required_files = [
    'outputs/km_survival_plot.png',
    'outputs/logrank_results.json',
    'outputs/cox_summary.csv',
    'outputs/cox_ph_test.json',
    'outputs/rsf_importance.png',
    'outputs/rsf_metrics.json',
]

print("Checking required outputs:\n")
all_exist = True
for filepath in required_files:
    exists = Path(filepath).exists()
    status = "✅" if exists else "❌"
    print(f"{status} {filepath}")
    all_exist = all_exist and exists

print("\n" + "="*50)
if all_exist:
    print("✅ ALL OUTPUTS GENERATED")
    print("Ready to push to GitHub!")
else:
    print("❌ MISSING OUTPUTS")
    print("Re-run the cells above to generate missing files.")

## Next Steps

1. ✅ Complete `AI_DISCLOSURE.md`
2. ✅ Run `python validate_submission.py` locally
3. ✅ Push to GitHub
4. ✅ Check that GitHub Actions tests pass
5. ✅ Upload outputs to Blackboard (if required)

Good luck! 🎉